# 01 Data Cleaning

This notebook prepares the experimental AWGC datasets for machine-learning analysis.

The workflow includes:

- Loading raw phase-composition datasets
- Merging initial phase composition with SBF phase evolution
- Creating engineered features
- Saving a processed dataset for machine learning


In [ ]:
import pandas as pd
from pathlib import Path

ROOT_DIR = Path('..')
RAW_DATA_DIR = ROOT_DIR / 'data' / 'raw'
PROCESSED_DATA_DIR = ROOT_DIR / 'data' / 'processed'

initial_path = RAW_DATA_DIR / 'initial_phase_composition.csv'
sbf_path = RAW_DATA_DIR / 'sbf_phase_evolution.csv'

initial_df = pd.read_csv(initial_path)
sbf_df = pd.read_csv(sbf_path)

initial_df.head(), sbf_df.head()

## Rename Columns

The initial and SBF datasets contain similar phase names. To avoid confusion after merging, the columns are renamed with `initial_` and `sbf_` prefixes.

In [ ]:
initial_df = initial_df.rename(columns={
    'amorphous_pct': 'initial_amorphous_pct',
    'wollastonite_pct': 'initial_wollastonite_pct',
    'hydroxyapatite_pct': 'initial_hydroxyapatite_pct',
    'whitlockite_pct': 'initial_whitlockite_pct'
})

sbf_df = sbf_df.rename(columns={
    'amorphous_pct': 'sbf_amorphous_pct',
    'wollastonite_pct': 'sbf_wollastonite_pct',
    'hydroxyapatite_pct': 'sbf_hydroxyapatite_pct',
    'whitlockite_pct': 'sbf_whitlockite_pct'
})

initial_df.head(), sbf_df.head()

## Merge Datasets

The initial phase-composition dataset is merged with the SBF phase-evolution dataset using sintering temperature as the common variable.

In [ ]:
ml_df = sbf_df.merge(initial_df, on='temperature_C', how='left')

ml_df.head()

## Feature Engineering

New features are created to support interpretable machine-learning analysis.

In [ ]:
ml_df['initial_crystallinity_pct'] = 100 - ml_df['initial_amorphous_pct']
ml_df['sbf_crystallinity_pct'] = 100 - ml_df['sbf_amorphous_pct']

ml_df['hydroxyapatite_gain_pct'] = (
    ml_df['sbf_hydroxyapatite_pct'] - ml_df['initial_hydroxyapatite_pct']
)

ml_df['wollastonite_change_pct'] = (
    ml_df['sbf_wollastonite_pct'] - ml_df['initial_wollastonite_pct']
)

ml_df.head()

## Save Processed Dataset

The final processed dataset is saved to `data/processed/awgc_ml_dataset.csv`.

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DATA_DIR / 'awgc_ml_dataset.csv'
ml_df.to_csv(output_path, index=False)

output_path